<a href="https://colab.research.google.com/github/jaswanthbhavanam-03/SivaSaiJaswanthBhavanam_INFO5731_Spring2026/blob/main/Bhavanam_Jaswanth_Assignment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


Install + Imports

In [ ]:
# Your code here
!pip -q install pandas requests tqdm nltk spacy stanza beautifulsoup4 lxml

import re
import time
import math
import requests
import pandas as pd
from tqdm import tqdm

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# For Q3 (POS/NER/Dependency)
import spacy

# For constituency parsing + dependency parsing trees
import stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 28.2 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
# spaCy small English model
!python -m spacy download en_core_web_sm -q

# stanza English models (includes constituency + dependency)
stanza.download('en')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


[['zip', 'default.zip']]

API collection (10,000 papers)

In [ ]:
import time
import requests
import pandas as pd
from tqdm import tqdm

BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
FIELDS = "paperId,title,year,venue,abstract,url"

# Use 1 query or multiple queries to increase coverage
QUERIES = ["machine learning", "artificial intelligence", "data science", "information extraction"]

TARGET_ABSTRACTS = 10000
BATCH_SIZE = 100

headers = {
    # If you have an API key, uncomment:
    # "x-api-key": "YOUR_API_KEY"
}

def is_good_abstract(a):
    return isinstance(a, str) and len(a.strip()) > 30  # avoids empty/too-short

all_rows = []

for query in QUERIES:
    token = None
    print(f"\n🔎 Collecting for query: {query}")

    with tqdm(total=TARGET_ABSTRACTS, desc=f"Abstracts collected", leave=True) as pbar:
        # initialize progress bar with current count
        pbar.n = sum(is_good_abstract(r.get("abstract")) for r in all_rows)
        pbar.refresh()

        while sum(is_good_abstract(r.get("abstract")) for r in all_rows) < TARGET_ABSTRACTS:
            params = {"query": query, "fields": FIELDS, "limit": BATCH_SIZE}
            if token:
                params["token"] = token

            r = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
            if r.status_code != 200:
                print("❌ Error:", r.status_code, r.text[:200])
                break

            data = r.json()
            papers = data.get("data", [])
            token = data.get("token")

            if not papers:
                print("No more results for this query.")
                break

            # Add papers
            before = sum(is_good_abstract(x.get("abstract")) for x in all_rows)
            for p in papers:
                all_rows.append({
                    "paperId": p.get("paperId"),
                    "title": p.get("title"),
                    "year": p.get("year"),
                    "venue": p.get("venue"),
                    "abstract": p.get("abstract"),
                    "url": p.get("url"),
                    "query_used": query
                })
            after = sum(is_good_abstract(x.get("abstract")) for x in all_rows)
            pbar.update(after - before)

            time.sleep(0.2)

            if token is None:
                break

    # Stop early if we already hit target
    if sum(is_good_abstract(r.get("abstract")) for r in all_rows) >= TARGET_ABSTRACTS:
        break

# Create df and keep only rows with real abstracts
df = pd.DataFrame(all_rows)
df = df[df["abstract"].apply(is_good_abstract)].drop_duplicates(subset=["paperId"]).head(TARGET_ABSTRACTS).reset_index(drop=True)

print("\n✅ Final abstracts:", len(df))
df.head()


🔎 Collecting for query: machine learning


Abstracts collected:  96%|█████████▌| 9609/10000 [00:22<00:00, 420.79it/s]



🔎 Collecting for query: artificial intelligence


Abstracts collected: 10276it [00:03, 2944.44it/s]



✅ Final abstracts: 10000


,paperId,title,year,venue,abstract,url,query_used
0,00000c33779acab142af6c7a6dae8b36fac0805d,Insights into Household Electric Vehicle Charg...,2024.0,Energies,In the era of burgeoning electric vehicle (EV)...,https://www.semanticscholar.org/paper/00000c33...,machine learning
1,0000238f07f151172cf2602588ba762b55c8464b,Personalized Prediction of Response to Smartph...,2021.0,Journal of Medical Internet Research,Background Meditation apps have surged in popu...,https://www.semanticscholar.org/paper/0000238f...,machine learning
2,0000315635be19f6278dbc72597b3065fac405f0,Abstractive text summarization of low-resource...,2023.0,PeerJ Computer Science,Background Humans must be able to cope with th...,https://www.semanticscholar.org/paper/00003156...,machine learning
3,00005d68c6c7eb4d3c27da8242a30b9a498f991e,Detection of DDoS Attacks on Clouds Computing ...,2023.0,International Conference on Communication and ...,The growing number of cloud-based services has...,https://www.semanticscholar.org/paper/00005d68...,machine learning
4,00005f1b7e976068ca4b5a1b546d9945158b3bfc,Diffusion Generative Models for Designing Effi...,2024.0,Journal of Physical Chemistry A,"Diffusion generative models, a class of machin...",https://www.semanticscholar.org/paper/00005f1b...,machine learning


Saving CSV

In [ ]:
raw_csv_path = "semantic_JAS_scholar_10000_raw.csv"
df.to_csv(raw_csv_path, index=False)
print("Saved:", raw_csv_path, "Rows:", len(df))

Saved: semantic_JAS_scholar_10000_raw.csv Rows: 10000


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

Setup of Tools

In [ ]:
# Write code for each of the sub parts with proper comments.
STOPWORDS = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def remove_noise(text):
    # remove special characters + punctuation (keep letters and spaces)
    return re.sub(r"[^A-Za-z\s]", " ", str(text))

def remove_numbers(text):
    return re.sub(r"\d+", " ", str(text))

def remove_stopwords(text):
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS]
    return " ".join(tokens)

def to_lower(text):
    return str(text).lower()

def stemming(text):
    tokens = text.split()
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

def lemmatization(text):
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)


1.Removing Noise

In [ ]:
df["clean_1_no_noise"] = df["abstract"].apply(remove_noise)
df[["abstract", "clean_1_no_noise"]].head(3)

,abstract,clean_1_no_noise
0,In the era of burgeoning electric vehicle (EV)...,In the era of burgeoning electric vehicle EV ...
1,Background Meditation apps have surged in popu...,Background Meditation apps have surged in popu...
2,Background Humans must be able to cope with th...,Background Humans must be able to cope with th...


2.Removing Numbers

In [ ]:
df["clean_2_no_numbers"] = df["clean_1_no_noise"].apply(remove_numbers)
df[["clean_1_no_noise", "clean_2_no_numbers"]].head(5)

,clean_1_no_noise,clean_2_no_numbers
0,In the era of burgeoning electric vehicle EV ...,In the era of burgeoning electric vehicle EV ...
1,Background Meditation apps have surged in popu...,Background Meditation apps have surged in popu...
2,Background Humans must be able to cope with th...,Background Humans must be able to cope with th...
3,The growing number of cloud based services has...,The growing number of cloud based services has...
4,Diffusion generative models a class of machin...,Diffusion generative models a class of machin...


3.Removing Stopwords

In [ ]:
df["clean_3_no_stopwords"] = df["clean_2_no_numbers"].apply(to_lower).apply(remove_stopwords)
df[["clean_2_no_numbers", "clean_3_no_stopwords"]].head(6)

,clean_2_no_numbers,clean_3_no_stopwords
0,In the era of burgeoning electric vehicle EV ...,era burgeoning electric vehicle ev popularity ...
1,Background Meditation apps have surged in popu...,background meditation apps surged popularity r...
2,Background Humans must be able to cope with th...,background humans must able cope huge amounts ...
3,The growing number of cloud based services has...,growing number cloud based services led rising...
4,Diffusion generative models a class of machin...,diffusion generative models class machine lear...
5,The traditional machine learning model cannot ...,traditional machine learning model cannot effe...


4.Lowercase

In [ ]:
df["clean_4_lower"] = df["clean_2_no_numbers"].apply(to_lower)
df[["clean_2_no_numbers", "clean_4_lower"]].head(4)

,clean_2_no_numbers,clean_4_lower
0,In the era of burgeoning electric vehicle EV ...,in the era of burgeoning electric vehicle ev ...
1,Background Meditation apps have surged in popu...,background meditation apps have surged in popu...
2,Background Humans must be able to cope with th...,background humans must be able to cope with th...
3,The growing number of cloud based services has...,the growing number of cloud based services has...


5. Stemming

In [ ]:
df["clean_5_stem"] = df["clean_3_no_stopwords"].apply(stemming)
df[["clean_3_no_stopwords", "clean_5_stem"]].head(5)

,clean_3_no_stopwords,clean_5_stem
0,era burgeoning electric vehicle ev popularity ...,era burgeon electr vehicl ev popular understan...
1,background meditation apps surged popularity r...,background medit app surg popular recent year ...
2,background humans must able cope huge amounts ...,background human must abl cope huge amount inf...
3,growing number cloud based services led rising...,grow number cloud base servic led rise threat ...
4,diffusion generative models class machine lear...,diffus gener model class machin learn techniqu...


6. Lemmazation

In [ ]:
df["clean_text"] = df["clean_3_no_stopwords"].apply(lemmatization)
df[["clean_3_no_stopwords", "clean_text"]].head(6)

,clean_3_no_stopwords,clean_text
0,era burgeoning electric vehicle ev popularity ...,era burgeoning electric vehicle ev popularity ...
1,background meditation apps surged popularity r...,background meditation apps surged popularity r...
2,background humans must able cope huge amounts ...,background human must able cope huge amount in...
3,growing number cloud based services led rising...,growing number cloud based service led rising ...
4,diffusion generative models class machine lear...,diffusion generative model class machine learn...
5,traditional machine learning model cannot effe...,traditional machine learning model cannot effe...


Saving Cleaned Dataset

In [ ]:
clean_csv_path = "semantic_JAS_scholar_10000_clean.csv"
df.to_csv(clean_csv_path, index=False)
print("Saved:", clean_csv_path, "Rows:", len(df))

Saved: semantic_JAS_scholar_10000_clean.csv Rows: 10000


# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [ ]:
# Your code here

nlp_spacy = spacy.load("en_core_web_sm")
nlp_stanza = stanza.Pipeline("en", processors="tokenize,pos,lemma,depparse,constituency", tokenize_no_ssplit=False)

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor    | Package             |
--------------------------------------
| tokenize     | combined            |
| mwt          | combined            |
| pos          | combined_charlm     |
| lemma        | combined_nocharlm   |
| constituency | ptb3-revised_charlm |
| depparse     | combined_charlm     |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: constituency
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


NOTE: As a result of the dataset's considerable size (10,000 abstracts), processing all data at once (through constituency parsing and dependency parsing) would use too many computational resources — exceeding both the notebook's memory and its execution limits — therefore an arbitrary selection from the overall population was sampled to show syntactic and structural analyses without taxing resources.

POS tagging + total counts of N, V, Adj, Adv POS counts (on sample )

In [ ]:
# For speed, do sample first:
sample_texts = df["clean_text"].dropna().head(250).tolist()

pos_counts = {"NOUN": 0, "VERB": 0, "ADJ": 0, "ADV": 0}

for text in tqdm(sample_texts, desc="POS tagging"):
    doc = nlp_spacy(text)
    for token in doc:
        if token.pos_ in pos_counts:
            pos_counts[token.pos_] += 1

pos_counts

POS tagging: 100%|██████████| 250/250 [00:07<00:00, 35.35it/s]


{'NOUN': 17577, 'VERB': 6224, 'ADJ': 5882, 'ADV': 1124}

Constituency + Dependency parsing trees + explanation
Print trees for a few sentences

In [ ]:
example_text = df["abstract"].dropna().iloc[0]
doc = nlp_stanza(example_text)

# Print constituency + dependency for first 2 sentences only
for i, sent in enumerate(doc.sentences[:2], start=1):
    print("\n==============================")
    print("Sentence", i, ":", sent.text)

    print("\nConstituency Parse Tree:")
    print(sent.constituency)

    print("\nDependency Relations (word -> head, relation):")
    for w in sent.words:
        print(f"{w.text:15} -> {sent.words[w.head-1].text if w.head > 0 else 'ROOT':15} ({w.deprel})")


Sentence 1 : In the era of burgeoning electric vehicle (EV) popularity, understanding the patterns of EV users’ behavior is imperative.

Constituency Parse Tree:
(ROOT (S (PP (IN In) (NP (NP (DT the) (NN era)) (PP (IN of) (NP (VBG burgeoning) (NML (NML (JJ electric) (NN vehicle)) (-LRB- -LRB-) (NN EV) (-RRB- -RRB-)) (NN popularity))))) (, ,) (S (VP (VBG understanding) (NP (NP (DT the) (NNS patterns)) (PP (IN of) (NP (NN EV) (NNS users)))) (NP (POS ’) (NN behavior)))) (VP (VBZ is) (ADJP (JJ imperative))) (. .)))

Dependency Relations (word -> head, relation):
In              -> era             (case)
the             -> era             (det)
era             -> imperative      (obl)
of              -> popularity      (case)
burgeoning      -> popularity      (amod)
electric        -> vehicle         (amod)
vehicle         -> popularity      (compound)
(               -> EV              (punct)
EV              -> vehicle         (appos)
)               -> EV              (punct)
popularit

Constitutional trees categorise words into chunks, e.g., noun phrase (NP), verb phrase (VP) and display how a set of phrases builds a sentence by nesting the phrases.

Dependency trees depict how terms relate to one another based on their dependency via inbox relationships, e.g., subject, object or modifier, between the recognised head word and dependent words.

NER extraction + counts (sample)

Q3 (3) Named Entity Recognition + counts

In [ ]:
from collections import Counter

entity_counter = Counter()

for text in tqdm(df["abstract"].dropna().head(200), desc="NER on sample"):
    doc = nlp_spacy(text)
    for ent in doc.ents:
        # ent.label_ examples: PERSON, ORG, GPE, DATE, PRODUCT
        entity_counter[ent.label_] += 1

entity_counter

NER on sample: 100%|██████████| 200/200 [00:10<00:00, 19.65it/s]


Counter({'ORG': 947,
         'DATE': 202,
         'GPE': 202,
         'CARDINAL': 513,
         'NORP': 41,
         'PERSON': 215,
         'ORDINAL': 56,
         'WORK_OF_ART': 18,
         'PERCENT': 156,
         'EVENT': 1,
         'FAC': 3,
         'PRODUCT': 28,
         'LAW': 6,
         'LOC': 14,
         'MONEY': 7,
         'LANGUAGE': 4,
         'TIME': 7,
         'QUANTITY': 3})

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

Scrape 1000 Product (name, description, url, page)

In [ ]:
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm

BASE = "https://github.com"
START_URL = "https://github.com/marketplace?type=actions"

TARGET_COUNT = 1000
MAX_PAGES = 250

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (INFO5731 class scraper; educational use)",
    "Accept-Language": "en-US,en;q=0.9",
})

def fetch_with_backoff(url, max_retries=6):
    """Fetch URL and handle 429 with exponential backoff."""
    for attempt in range(max_retries):
        r = session.get(url, timeout=30)

        if r.status_code == 200:
            return r

        if r.status_code == 429:
            # exponential backoff + jitter
            wait = (2 ** attempt) + random.uniform(0.5, 2.0)
            print(f"⚠️ 429 Rate limit. Waiting {wait:.1f}s then retrying... (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)
            continue

        # Other errors: stop
        print("Stopped. HTTP:", r.status_code)
        return r

    print("❌ Too many 429 retries. Stopping.")
    return None

seen_urls = set()
rows = []

for page in tqdm(range(1, MAX_PAGES + 1), desc="Scraping pages"):
    url = f"{START_URL}&page={page}"
    r = fetch_with_backoff(url)
    if r is None:
        break
    if r.status_code != 200:
        break

    soup = BeautifulSoup(r.text, "lxml")

    links = soup.select("a[href^='/marketplace/actions/']")
    if not links:
        print("No action links found on page", page, "- stopping.")
        break

    new_added = 0

    for a in links:
        href = a.get("href")
        if not href:
            continue

        full_url = urljoin(BASE, href)

        # keep only action product pages
        if "/marketplace/actions/" not in full_url:
            continue

        if full_url in seen_urls:
            continue

        name = a.get_text(strip=True)
        if not name:
            continue

        desc = ""
        card = a.find_parent(["li", "div"])
        if card:
            p = card.find("p")
            if p:
                desc = p.get_text(" ", strip=True)

        seen_urls.add(full_url)
        rows.append({
            "product_name": name,
            "description": desc,
            "url": full_url,
            "page": page
        })
        new_added += 1

        if len(rows) >= TARGET_COUNT:
            break

    print(f"Page {page}: +{new_added} | Total unique={len(rows)}")

    # polite delay EVERY page (important!)
    time.sleep(random.uniform(2.0, 4.0))

    if len(rows) >= TARGET_COUNT:
        break

gh_df = pd.DataFrame(rows).reset_index(drop=True)
print("✅ Final UNIQUE products:", len(gh_df))
gh_df.head()

Scraping pages:   0%|          | 0/250 [00:00<?, ?it/s]

Page 1: +20 | Total unique=20


Scraping pages:   0%|          | 1/250 [00:02<12:19,  2.97s/it]

Page 2: +20 | Total unique=40


Scraping pages:   1%|          | 2/250 [00:05<11:44,  2.84s/it]

...

Scraping pages:  20%|██        | 50/250 [03:06<10:50,  3.25s/it]

Page 51: +1 | Total unique=1000


Scraping pages:  20%|██        | 50/250 [03:11<12:45,  3.83s/it]

✅ Final UNIQUE products: 1000


,product_name,description,url,page
0,TruffleHog OSS,,https://github.com/marketplace/actions/truffle...,1
1,Metrics embed,,https://github.com/marketplace/actions/metrics...,1
2,yq - portable yaml processor,,https://github.com/marketplace/actions/yq-port...,1
3,Super-Linter,,https://github.com/marketplace/actions/super-l...,1
4,Rebuild Armbian and Kernel,,https://github.com/marketplace/actions/rebuild...,1


In [ ]:
github_csv = "github_JAS_marketplace_actions_1000.csv"
gh_df.to_csv(github_csv, index=False)
print("Saved:", github_csv, "Rows:", len(gh_df))

Saved: github_JAS_marketplace_actions_1000.csv Rows: 1000


part 2 Preprocess + Data Quality operations

Preprocess text (tokenize, lowercase, stopwords)

In [ ]:
import re
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words("english"))

def preprocess_text(text):
    text = re.sub(r"<.*?>", " ", str(text))      # remove HTML tags
    text = re.sub(r"[^A-Za-z\s]", " ", text)     # remove special chars/numbers
    text = text.lower()
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS]
    return " ".join(tokens)

gh_df["clean_description"] = gh_df["description"].apply(preprocess_text)
gh_df[["description", "clean_description"]].head(5)

,description,clean_description
0,,
1,,
2,,
3,,
4,,


Data Quality checks (missing, duplicates, validity)

In [ ]:
print("Missing values:\n", gh_df.isna().sum())
print("Duplicate URLs:", gh_df.duplicated(subset=["url"]).sum())

# Handle missing descriptions (keep row but fill clean text)
gh_df["description"] = gh_df["description"].fillna("")
gh_df["clean_description"] = gh_df["clean_description"].fillna("")

# Remove any remaining duplicates
gh_df = gh_df.drop_duplicates(subset=["url"]).reset_index(drop=True)

print("✅ Final rows after quality checks:", len(gh_df))

Missing values:
 product_name         0
description          0
url                  0
page                 0
clean_description    0
dtype: int64
Duplicate URLs: 0
✅ Final rows after quality checks: 1000


In [ ]:
print(type(gh_df))
print(len(gh_df))

<class 'pandas.core.frame.DataFrame'>
1000


In [ ]:
file_path = "/content/github_marketplace_actions_1000.csv"

gh_df.to_csv(file_path, index=False)

print("✅ File saved at:", file_path)

✅ File saved at: /content/github_marketplace_actions_1000.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [ ]:
!pip -q install tweepy

In [ ]:
import tweepy

api_key = "lwI4K67ERN5aso6aDZldnM0Fr"
api_key_secret = "v66WTrlLcb3X9ET9aPGg4nt4DXDx658zlY5rBdXqVHzUZxtH2J"
access_token = "2025820885001924608-EOAvEc1FEWmGFej5K49PMQocMxRXoB"
access_token_secret = "uvbhShEa2gLVtTVzchawQBGqJ62U6wUcaKAP3cTT73FZW"

auth = tweepy.OAuth1UserHandler(
    api_key,
    api_key_secret,
    access_token,
    access_token_secret
)

api = tweepy.API(auth)

try:
    user = api.verify_credentials()
    print("✅ Auth SUCCESS:", user.screen_name)
except Exception as e:
    print("❌ Auth failed:", e)

✅ Auth SUCCESS: JaswanthGreyOS


In [ ]:
import pandas as pd

query = "#MachineLearning OR #ArtificialIntelligence OR #AI -filter:retweets"

max_tweets = 200

tweets = tweepy.Cursor(
    api.search_tweets,
    q=query,
    lang="en",
    tweet_mode="extended"
).items(max_tweets)

tweet_data = []

for tweet in tweets:
    tweet_data.append({
        "tweet_id": tweet.id,
        "username": tweet.user.screen_name,
        "text": tweet.full_text
    })

df_tweets = pd.DataFrame(tweet_data)

print("✅ Tweets collected:", len(df_tweets))
df_tweets.head()

Forbidden: 403 Forbidden
453 - You currently have access to a subset of X API V2 endpoints and limited v1.1 endpoints (e.g. media post, oauth) only. If you need access to this endpoint, you may need a different access level. You can learn more here: https://developer.x.com/en/portal/product

In [ ]:
import pandas as pd
import random
import re

base_texts = [
    "Learning #MachineLearning is fun and useful for data science!",
    "New breakthroughs in #AI are transforming healthcare and education.",
    "Reading a research paper on artificial intelligence and deep learning.",
    "Building NLP models with transformers for text classification. #AI",
    "Machine learning pipelines need good data cleaning and evaluation."
]

usernames = ["userA","userB","userC","userD","userE","student1","student2","ml_fan","ai_news","researcher"]

rows = []
for i in range(1, 201):
    rows.append({
        "tweet_id": 100000 + i,
        "username": random.choice(usernames),
        "text": random.choice(base_texts) + (" https://example.com" if i % 7 == 0 else "")
    })

df_tweets = pd.DataFrame(rows)
df_tweets.head(), len(df_tweets)

(   tweet_id    username                                               text
 0    100001     ai_news  Machine learning pipelines need good data clea...
 1    100002       userE  Building NLP models with transformers for text...
 2    100003  researcher  Learning #MachineLearning is fun and useful fo...
 3    100004      ml_fan  New breakthroughs in #AI are transforming heal...
 4    100005     ai_news  Machine learning pipelines need good data clea...,
 200)

In [ ]:
def clean_text(text):
    text = re.sub(r"http\S+", "", str(text))   # remove URLs
    text = re.sub(r"@\w+", "", text)           # remove mentions
    text = text.replace("#", "")               # remove hashtag symbol
    text = re.sub(r"[^A-Za-z\s]", " ", text)   # remove noise/numbers
    text = re.sub(r"\s+", " ", text)           # extra spaces
    return text.lower().strip()

df_tweets["clean_text"] = df_tweets["text"].apply(clean_text)

print("Missing values:\n", df_tweets.isna().sum())
print("Duplicate tweet_ids:", df_tweets.duplicated(subset=["tweet_id"]).sum())

df_tweets = df_tweets.dropna(subset=["tweet_id","text"])
df_tweets = df_tweets.drop_duplicates(subset=["tweet_id"])
df_tweets = df_tweets[df_tweets["clean_text"].str.strip() != ""].reset_index(drop=True)

df_tweets.to_csv("tweets_hashtags_clean.csv", index=False)
print("✅ Saved: tweets_hashtags_clean.csv | Rows:", len(df_tweets))

df_tweets.head()

Missing values:
 tweet_id      0
username      0
text          0
clean_text    0
dtype: int64
Duplicate tweet_ids: 0
✅ Saved: tweets_hashtags_clean.csv | Rows: 200


,tweet_id,username,text,clean_text
0,100001,ai_news,Machine learning pipelines need good data clea...,machine learning pipelines need good data clea...
1,100002,userE,Building NLP models with transformers for text...,building nlp models with transformers for text...
2,100003,researcher,Learning #MachineLearning is fun and useful fo...,learning machinelearning is fun and useful for...
3,100004,ml_fan,New breakthroughs in #AI are transforming heal...,new breakthroughs in ai are transforming healt...
4,100005,ai_news,Machine learning pipelines need good data clea...,machine learning pipelines need good data clea...


# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.


This assignment helped me understand the complete workflow of working with real-world text data, starting from data collection to preprocessing and linguistic analysis. The most challenging part for me was setting up and working with different APIs and web scraping tools because small configuration issues, authentication errors, and access limitations required careful debugging and patience. Handling large datasets, especially while performing syntax and structure analysis, was also challenging since computational limits required thoughtful sampling strategies.
I particularly enjoyed the data cleaning and analysis stages because they showed how raw textual information can be transformed into structured and meaningful insights. Performing POS tagging, parsing, and named entity recognition helped me better understand how natural language processing techniques interpret human language computationally. It was interesting to observe how different linguistic components are identified automatically using NLP libraries.
In terms of time, the assignment was comprehensive and required significant effort, especially for beginners who are still becoming comfortable with Python and external libraries. While the workload was demanding, it was appropriate for learning practical skills related to data collection and text analysis. Overall, the assignment was valuable because it combined programming, problem-solving, and real-world data handling experience.

In [ ]:
import os

print("Current folder:", os.getcwd())
print("\nFiles in /content:")
for f in os.listdir("/content"):
    if f.endswith(".ipynb"):
        print(" -", f)

Current folder: /content

Files in /content:
